# LLM Cycle-Geometry — Task 15 Full-Model Run (Colab)

Runs Part 1 (circle reproduction, the validation gate) and Part 2 (the AUC comparison ladder) against the real `google/gemma-2-2b` model.

**Before running:** Runtime → Change runtime type → select a GPU (e.g. T4).

## 1. Mount Google Drive and install the package wheel

The wheel (`networkgeometry-0.1.2-py3-none-any.whl`) lives in your Drive folder `NetworkGeometry-Colab/`. This mounts your Drive (one browser authorization click) and installs directly from there — no manual upload needed.

**0.1.2 fixes a real bug, not just infra:** every prompt template previously ended with a period after `{X}` (or, in one case, put `{X}` first). Confirmed against Gemma's tokenizer, that period becomes its own final token — so `run_part1`/`run_part2` were reading the activation of a period, not the state word, for essentially every prompt in the whole pipeline. Templates are now state-word-final with no trailing punctuation (see spec §3.4). **Any results from wheels ≤0.1.1 should be discarded and re-run.**

**When you rebuild the wheel locally after a code fix, the version number bumps (0.1.0 → 0.1.1 → 0.1.2 ...) so the filename always changes.** Delete the old versioned wheel from the Drive folder and drop in the new one — never overwrite a same-named file, so there's no ambiguity about whether Drive actually has the latest fix.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/NetworkGeometry-Colab'
WHEEL_PATH = f'{DRIVE_DIR}/networkgeometry-0.1.2-py3-none-any.whl'
print('Wheel path:', WHEEL_PATH)

In [ ]:
import numpy, scipy
NUMPY_VER = numpy.__version__
SCIPY_VER = scipy.__version__
print(f'Pinning Colab stock numpy=={NUMPY_VER} scipy=={SCIPY_VER} so nothing upgrades them')

!pip install -q "{WHEEL_PATH}" "numpy=={NUMPY_VER}" "scipy=={SCIPY_VER}" transformer_lens
!pip uninstall -y -q torchvision
!pip uninstall -y -q torchaudio

import importlib.metadata
installed_ver = importlib.metadata.version('networkgeometry')
expected_ver = WHEEL_PATH.split('networkgeometry-')[1].split('-py3')[0]
assert installed_ver == expected_ver, (
    f'Installed networkgeometry=={installed_ver} but expected =={expected_ver} — '
    'the wheel on Drive is stale, re-upload it.'
)
print(f'Confirmed networkgeometry=={installed_ver} installed')

**Root cause of the numpy/scipy crash, actually found:** `pip install transformer_lens` doesn't go through this project's lockfile at all — it's a fresh PyPI resolution every time, independent of the relaxed floors in `pyproject.toml` (those only govern our own wheel's declared requirements, which are already satisfied and never force an upgrade). If `transformer_lens`/`transformers` on PyPI have since bumped their own numpy/scipy floors, installing them can silently upgrade Colab's preinstalled numpy to a version `sklearn` (also preinstalled, imported transitively by `transformers.generation.candidate_generator`) wasn't built against, causing `AttributeError: ... has no attribute '_blas_supports_fpe'`. This is why the fix kept "working" on one VM and breaking again on the next.

The cell above now captures Colab's stock numpy/scipy versions before installing anything, then pins those exact versions in the same `pip install` call as `transformer_lens` — so pip's resolver can't upgrade them no matter what transformer_lens's own metadata requests, keeping Colab's internally-consistent preinstalled numpy/scipy/sklearn trio intact.

Two follow-on version-skew issues also showed up, both fixed by the cell above:

- `torchvision::nms` op-registration failure — Colab's preinstalled `torchvision` doesn't match whatever `torch` gets installed alongside `transformer_lens`. We don't need vision utilities for a text-only Gemma model, so the cell above uninstalls `torchvision` outright.
- `RuntimeError: PyTorch and TorchAudio were compiled with different CUDA versions` — same story for `torchaudio`, which we also don't need. Uninstalled too.

**Important:** any time you change runtime type (GPU, RAM tier, etc.), Colab gives you a brand-new VM with none of this applied — you must re-run the cell above from scratch on that new VM. A plain Runtime to Restart session (no runtime-type change) keeps the same VM/disk, so pip state survives; you'd only need to restart if you still have transformers/torch imported in memory from a prior attempt in this session (stale in-memory "is this package available" caches won't reflect an uninstall until the process restarts).

## 2. Authenticate with Hugging Face

Runs an interactive login widget — paste your token (the same one used locally, or a fresh one). You must have already accepted the license at https://huggingface.co/google/gemma-2-2b under this account.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from huggingface_hub import model_info
info = model_info('google/gemma-2-2b')
print('Access confirmed:', info.id, '| gated:', info.gated)

## 3. Load the model (GPU, no-processing loading path, fp16)

The standard `from_pretrained` path (with TransformerLens's weight processing — LayerNorm folding, `center_writing_weights`, `center_unembed`) crashed the whole Colab session with an out-of-memory error during weight conversion, even on GPU — separately from the local Windows/CPU crash. It also can't cleanly apply `center_unembed` to Gemma-2 anyway (TransformerLens warns that `center_unembed` isn't valid with logit softcapping, which Gemma-2 uses).

So this cell uses `from_pretrained_no_processing` (skips the memory-heavy folding/centering steps) with `dtype=torch.float16` (halves peak RAM versus the default float32 — Gemma's checkpoint is natively bfloat16 on disk anyway). Per the documented methodological caveat, note in the findings memo that the no-processing loading path was used: it skips `center_writing_weights`, which zeroes a residual-stream direction that subsequent LayerNorms discard anyway — a small, likely-negligible deviation, but worth recording.

If this still OOMs, the remaining lever is a higher-RAM Colab runtime (Runtime → Change runtime type) — a paid decision, not something to reach for automatically.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())

from transformer_lens import HookedTransformer
import time
t0 = time.perf_counter()
model = HookedTransformer.from_pretrained_no_processing(
    'google/gemma-2-2b',
    device='cuda' if torch.cuda.is_available() else 'cpu',
    dtype=torch.float16,
)
print(f'Model loaded in {time.perf_counter()-t0:.1f}s')

## 4. Run Part 1 — circle reproduction (the validation gate)

Per spec §4.5: if month/day circularity doesn't reproduce cleanly here, treat it as a pipeline bug and debug before trusting Part 2 — do not proceed on a weak Part 1 result.

In [ ]:
from networkgeometry.run import run_part1
import time

LAYERS = list(range(26))
OUT_DIR = 'results'

t0 = time.perf_counter()
part1_results = run_part1(model, LAYERS, OUT_DIR)
print(f'Part 1 done in {time.perf_counter()-t0:.1f}s')

for name, layer_results in part1_results.items():
    print(f'\n=== {name} ===')
    for lc in layer_results:
        print(f'  layer {lc.layer:2d}: residual={lc.normalized_residual:.4f}  '
              f'angular_order={lc.angular_order:.4f}  top2_var_ratio={lc.top2_variance_ratio:.4f}')

**Stop and inspect before continuing.** Look for layers where `angular_order` and `top2_variance_ratio` are both high (>0.9-ish) for both day and month — those are the clean, reproduced-circle layers. `results/part1_summary.json` now has these numbers on disk too.

## 5. Plot Part 1 manifolds (optional, for visual inspection)

In [ ]:
from networkgeometry.figures.part1_plots import plot_circularity_by_layer
import matplotlib.pyplot as plt
from IPython.display import Image, display

for name, layer_results in part1_results.items():
    path = plot_circularity_by_layer(layer_results, f'{OUT_DIR}/{name}_circularity_by_layer.png')
    display(Image(filename=path))

## 6. Run Part 2 — the AUC comparison ladder

Set `LAYERS` below to the gate-passing layers you identified in Part 1 (or leave as the same full range — `run_part2` applies its own within-structure gate internally per layer, so passing all 26 is safe, just slower).

In [ ]:
from networkgeometry.run import run_part2

t0 = time.perf_counter()
part2_results = run_part2(model, LAYERS, OUT_DIR)
print(f'Part 2 done in {time.perf_counter()-t0:.1f}s')

for lr in part2_results:
    print(f'\nlayer {lr.layer}: within={lr.within:.4f} gate_passed={lr.gate_passed}')
    for target, stats in lr.crosses.items():
        print(f'  -> {target}: auc={stats["auc"]:.4f} sem={stats["sem"]:.4f} '
              f'raw_p={stats["raw_p"]:.4f} fdr_p={stats["fdr_p"]:.4f} bonferroni_p={stats["bonferroni_p"]:.4f}')

`results/summary.json` now has the full ladder on disk.

## 7. Plot Part 2 results

In [ ]:
from networkgeometry.figures.part2_plots import plot_auc_by_layer

path = plot_auc_by_layer(part2_results, ['month', 'years', 'hierarchy', 'flat'], f'{OUT_DIR}/auc_by_layer.png')
display(Image(filename=path))

## 8. Download results back to your machine

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('task15_results', 'zip', OUT_DIR)
files.download('task15_results.zip')

# Also persist a copy to the same Drive folder
shutil.copy('task15_results.zip', f'{DRIVE_DIR}/task15_results.zip')
print(f'Results also saved to Drive: {DRIVE_DIR}/task15_results.zip')